In [1]:
# Install the needed dependeincies
# !pip install requests numpy polars pandas

# HDD analysis

Load the needed libraries

In [2]:
import re
import subprocess
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import polars as pl

# general settings for polars
pl.Config.set_tbl_cols(1000)
pl.Config.set_tbl_width_chars(1000)

polars.config.Config

In [3]:
# ---- Constants
BACKENDS = ["POSIX", "MPIIO"]
SCENARIOS=(
	"write_seq_4k",
	"read_seq_4k",
	"write_seq_4m",
	"read_seq_4m",
	"write_rand_4k",
	"read_rand_4k",
	"writeread_rand_4k_shared",
)
NITER = 10
SLURM_JOB_ID_SCRATCH = "955566"

# offset to apply before the first timestamp and after the last timestamp performing the query,
#  needed in plots
OFFSET_IN_SECONDS = 60 

In [4]:
# --- CONSTANTS
PRJ_ROOT_DIR = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"],
        text=True
    ).strip()
)
DATA_DIR = Path(PRJ_ROOT_DIR, "data")
IOR_RESULTS_DIR = PRJ_ROOT_DIR / "03-ior-bench" / "results" / "GENOA" / "scratch"

# print(f"Project root directory: {PRJ_ROOT_DIR}")
# print(f"Data directory: {DATA_DIR}")
# print(f"Ior results directory: {IOR_RESULTS_DIR}")

### Load the experiments table 

We are going to use it as a support table to simplify analysis

In [5]:
experiment_timetable = pl.read_parquet(DATA_DIR / "experiments" / "experiments.parquet")
experiment_timetable = experiment_timetable.filter(pl.col("SLURM_JOB_ID") == SLURM_JOB_ID_SCRATCH)
# experiment_timetable = experiment_timetable.filter(pl.col("SLURM_JOB_ID") == SLURM_JOB_ID_FAST)
experiment_timetable = experiment_timetable.select(["timestamp", "status", "notes"])
host_nodes = ("genoa007", "genoa008")

# print(experiment_timetable)
# print(time_min)
# print(time_max)

Reshape the experiment table in a more easy-to-tract dictionary

In [6]:
#  ---- 1. Parse fields from timetable -----
parsed = (
    experiment_timetable
    .with_columns(
        # pl.col("timestamp").str.strptime(
        #     pl.Datetime(time_zone="UTC"),
        #     format="%Y-%m-%dT%H:%M:%S%.fZ",
        #     strict=False,
        # ),
        pl.col("notes")
        .str.extract(r"backend=([^-]+)", 1)
        .str.to_uppercase()
        .alias("backend"),

        pl.col("notes")
        .str.extract(r"scenario=(.+)_([0-9]+)$", 1)
        .alias("scenario"),

        pl.col("notes")
        .str.extract(r"scenario=(.+)_([0-9]+)$", 2)
        .cast(pl.Int64)
        .alias("iter"),
    )
    .filter(
        pl.col("backend").is_not_null()
        & pl.col("scenario").is_not_null()
        & pl.col("iter").is_not_null()
    )
)


#  ----- 2. Reshape START / END into columns ------
runs = (
    parsed
    .pivot(
        values="timestamp",
        index=["backend", "scenario", "iter"],
        on="status",
        aggregate_function="first",
    )
    .rename({
        "START": "start",
        "END": "end",
    })
    .sort(["backend", "scenario", "iter"])
)


#  ------ 3. Build nested dictionary ------
timetables = {}

for backend in BACKENDS:
    timetables[backend] = {}

    for scenario in SCENARIOS:
        df = (
            runs
            .filter(
                (pl.col("backend") == backend)
                & (pl.col("scenario") == scenario)
            )
            .select(["iter", "start", "end"])
            .sort("iter")
        )

        if df.height > 0:
            timetables[backend][scenario] = df


# --- example for debug
# print(timetables["POSIX"]["write_seq_4k"])

### Load the data

In [7]:
# ----- Load the data from DATA_DIR
tables = ["ipmi_power","cpu","diskio","net","ipmi_sensor","turbostat","pdu_power"]
columns_to_keep = {
    "ipmi_power": ["timestamp", "host", "value", "unit", "psu_id"],
    "cpu": ["timestamp", "host", "core_id", "usage_user", "usage_system", "usage_iowait", "usage_idle"],
    "diskio": ["timestamp", "host", "name", "merged_writes","reads","writes","read_bytes","write_bytes","read_time","write_time","merged_reads","io_time","weighted_io_time"],
    "net": ["timestamp", "host", "interface", "bytes_sent", "bytes_recv", "packets_sent", "packets_recv"],
    "ipmi_sensor": ["timestamp", "host", "name", "value"],
    "turbostat": ["host", "timestamp", "average_frequency_mhz", "busy_frequency_mhz", "core", "cpu", "core_temperature_celsius", "package", "package_temperature_celsius", "uncore_frequency_mhz"],
    "pdu_power": ["timestamp", "connected_device", "value_W"],
}

data = {}

In [8]:
def read_data_from_parquet(table_name, timefilter=None, column_names=None, data_dir=DATA_DIR):
    parquet_files = sorted((data_dir / table_name).glob("*.parquet"))

    if not parquet_files:
        print(f"No parquet files found for table {table_name} in {data_dir / table_name}")
        return pl.DataFrame()

    if timefilter is not None:
        time_min, time_max = timefilter

        selected_files = []
        for file in parquet_files:
            parts = file.stem.split("_")
            file_start = datetime.fromisoformat(parts[-2]).replace(tzinfo=time_min.tzinfo)
            file_end = datetime.fromisoformat(parts[-1]).replace(tzinfo=time_min.tzinfo)

            if file_start <= time_max and file_end >= time_min:
                selected_files.append(file)

        parquet_files = selected_files

    if not parquet_files:
        print(f"No parquet files overlap requested timerange for table {table_name}")
        return pl.DataFrame()

    dfs = []
    for file in parquet_files:
        try:
            df = pl.read_parquet(file)
            if column_names:
                df = df.select(column_names)
            dfs.append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")

    if not dfs:
        print(f"No valid parquet files found for table {table_name} in {data_dir / table_name}")
        return pl.DataFrame()

    df = pl.concat(dfs, how="vertical")

    if table_name == "ipmi_power":
        df = df.with_columns(
            [
                pl.col("unit").cast(pl.Utf8).str.strip_chars(),
                pl.col("host").cast(pl.Utf8).str.strip_chars(),
                pl.col("psu_id").cast(pl.Int64, strict=False),
            ]
        ).filter(
            (pl.col("unit") == "W") &
            (pl.col("psu_id") == 0) &
            (pl.col("host").str.starts_with("ceph"))
        )

    if timefilter is not None:
        df = df.filter(pl.col("timestamp").is_between(timefilter[0], timefilter[1]))

    return df

In [9]:
for table in tables:
    data[table] = {}

    for backend in BACKENDS:
        if backend not in timetables:
            continue

        data[table][backend] = {}

        for scenario in SCENARIOS:
            if scenario not in timetables[backend]:
                continue

            time_min = (
                timetables[backend][scenario]
                .select(pl.col("start").min())[0, 0]
                - timedelta(seconds=OFFSET_IN_SECONDS)
            )

            time_max = (
                timetables[backend][scenario]
                .select(pl.col("end").max())[0, 0]
                + timedelta(seconds=OFFSET_IN_SECONDS)
            )

            data[table][backend][scenario] = read_data_from_parquet(
                table,
                column_names=columns_to_keep[table],
                timefilter=[time_min, time_max],
            )

            # print(f"processed table {table} - scenario {scenario} - backend {backend}")

In [10]:
# data

### Aggregate the data

In [11]:
def _ensure_utc_timestamp(df, col="timestamp"):
    if df.is_empty():
        return df

    dtype = df.schema[col]

    if dtype == pl.String:
        return df.with_columns(
            pl.col(col)
            .str.to_datetime(strict=False, time_zone="UTC")
            .alias(col)
        )

    if str(dtype).startswith("Datetime"):
        tz = getattr(dtype, "time_zone", None)

        if tz is None:
            return df.with_columns(
                pl.col(col).dt.replace_time_zone("UTC").alias(col)
            )

        if tz != "UTC":
            return df.with_columns(
                pl.col(col).dt.convert_time_zone("UTC").alias(col)
            )

    return df


def calculate_io_power_efficiency(data_diskio, data_pw, timetable_df, bucket="1s"):
    data_diskio = _ensure_utc_timestamp(data_diskio, "timestamp")
    data_pw = _ensure_utc_timestamp(data_pw, "timestamp")
    timetable_df = _ensure_utc_timestamp(timetable_df, "timestamp")

    # --------------------------------------------
    # 1) Power data
    # --------------------------------------------
    pw_agg = (
        data_pw
        .with_columns(pl.col("timestamp").dt.truncate(bucket).alias("ts"))
        .group_by("ts")
        .agg(pl.col("value").sum().alias("total_power_W"))
        .sort("ts")
    )

    # --------------------------------------------
    # 2) Disk data + rates
    # --------------------------------------------
    disk_ts = (
        data_diskio
        .with_columns(pl.col("timestamp").dt.truncate(bucket).alias("ts"))
        .group_by("ts")
        .agg([
            pl.sum("read_bytes").alias("read_bytes"),
            pl.sum("write_bytes").alias("write_bytes"),
            pl.sum("reads").alias("reads"),
            pl.sum("writes").alias("writes"),
            pl.sum("merged_reads").alias("merged_reads"),
            pl.sum("merged_writes").alias("merged_writes"),
            pl.sum("read_time").alias("read_time"),
            pl.sum("write_time").alias("write_time"),
            pl.sum("io_time").alias("io_time"),
            pl.sum("weighted_io_time").alias("weighted_io_time"),
        ])
        .sort("ts")
    )

    disk_rate = (
        disk_ts
        .with_columns([
            pl.col("read_bytes").diff().clip(lower_bound=0).fill_null(0).alias("read_bytes_rate"),
            pl.col("write_bytes").diff().clip(lower_bound=0).fill_null(0).alias("write_bytes_rate"),
            pl.col("reads").diff().clip(lower_bound=0).fill_null(0).alias("reads_rate"),
            pl.col("writes").diff().clip(lower_bound=0).fill_null(0).alias("writes_rate"),
            pl.col("merged_reads").diff().clip(lower_bound=0).fill_null(0).alias("merged_reads_rate"),
            pl.col("merged_writes").diff().clip(lower_bound=0).fill_null(0).alias("merged_writes_rate"),
        ])
        .with_columns([
            (pl.col("read_bytes_rate") / (1024 * 1024)).alias("read_MBps"),
            (pl.col("write_bytes_rate") / (1024 * 1024)).alias("write_MBps"),
        ])
    )

    # --------------------------------------------
    # 3) Join + efficiency metrics
    # --------------------------------------------
    efficiency_df = (
        disk_rate
        .join(pw_agg, on="ts", how="inner")
        .with_columns([
            (pl.col("read_MBps") + pl.col("write_MBps")).alias("total_MBps"),
            (
                pl.col("reads_rate")
                + pl.col("writes_rate")
                + pl.col("merged_reads_rate")
                + pl.col("merged_writes_rate")
            ).alias("total_iops"),
        ])
        .with_columns([
            pl.when(pl.col("total_power_W") > 0)
              .then(pl.col("total_MBps") / pl.col("total_power_W"))
              .otherwise(None)
              .alias("MBps_per_watt"),
            pl.when(pl.col("total_power_W") > 0)
              .then(pl.col("total_iops") / pl.col("total_power_W"))
              .otherwise(None)
              .alias("iops_per_watt"),
        ])
    )

    if efficiency_df.is_empty():
        return efficiency_df.select(
            pl.lit(None).cast(pl.Float64).alias("mean_MBps_per_watt"),
            pl.lit(None).cast(pl.Float64).alias("mean_iops_per_watt"),
            pl.lit(None).cast(pl.Float64).alias("mean_MBps"),
            pl.lit(None).cast(pl.Float64).alias("mean_iops"),
            pl.lit(None).cast(pl.Float64).alias("mean_power_W"),
        )

    # --------------------------------------------
    # 4) Filter idle time using timetable
    # --------------------------------------------
    ts_min, ts_max = efficiency_df.select(
        pl.col("ts").min().alias("ts_min"),
        pl.col("ts").max().alias("ts_max"),
    ).row(0)

    events = (
        timetable_df
        .filter(pl.col("status").is_in(["START", "END"]))
        .sort("timestamp")
    )

    starts = (
        events
        .filter(pl.col("status") == "START")
        .select(pl.col("timestamp").alias("start"))
        .with_row_index("idx")
    )

    ends = (
        events
        .filter(pl.col("status") == "END")
        .select(pl.col("timestamp").alias("end"))
        .with_row_index("idx")
    )

    intervals = (
        starts
        .join(ends, on="idx", how="inner")
        .drop("idx")
        .filter(pl.col("end") >= pl.col("start"))
        .filter(
            (pl.col("start") <= ts_max) &
            (pl.col("end") >= ts_min)
        )
        .sort("start")
    )

    if intervals.height > 0:
        cond = pl.lit(False)
        for start, end in intervals.iter_rows():
            cond = cond | ((pl.col("ts") >= start) & (pl.col("ts") <= end))
        efficiency_df = efficiency_df.filter(cond)
    else:
        efficiency_df = efficiency_df.clear()

    # --------------------------------------------
    # 5) Summary
    # --------------------------------------------
    efficiency_summary = efficiency_df.select(
        pl.col("MBps_per_watt").mean().alias("mean_MBps_per_watt"),
        pl.col("iops_per_watt").mean().alias("mean_iops_per_watt"),
        pl.col("total_MBps").mean().alias("mean_MBps"),
        pl.col("total_iops").mean().alias("mean_iops"),
        pl.col("total_power_W").mean().alias("mean_power_W"),
    )

    return efficiency_summary


In [12]:
summary_results = []

for backend in BACKENDS:
    for scenario in SCENARIOS:
        if scenario in timetables[backend]:
            # print(f"Processing summary for {backend} - {scenario}...")
            
            stats_df = calculate_io_power_efficiency(
                data_diskio=data["diskio"][backend][scenario], 
                data_pw=data["ipmi_power"][backend][scenario], 
                timetable_df=experiment_timetable,
                bucket="1s"
            )            
            
            # Add literal columns so we know which backend/scenario these rows belong to
            stats_df = stats_df.with_columns([
                pl.lit(backend).alias("backend"),
                pl.lit(scenario).alias("scenario")
            ])
            summary_results.append(stats_df)

df_summary = pl.concat(summary_results, how="vertical")

# print everything
# with pl.Config(tbl_rows=-1):
#     print(df_summary)


### Parse results file 

Parse results file from the ior benchmark and check the performance in the various workloads

In [13]:
def parse_ior_summary(filepath):
    path = Path(filepath)
    filename = path.name
    job_id = SLURM_JOB_ID_SCRATCH
    fs_type = "scratch"  # hardcoded: this notebook currently analyzes scratch only
    
    # Filenames are like: read_rand_4k_mpiio_summary.txt_1
    match = re.match(r"(.*)_(mpiio|posix)_summary\.txt_(\d+)", filename)
    if not match:
        return None
    
    scenario = match.group(1)
    api = match.group(2).upper()
    run_id = int(match.group(3))
    
    bw = None
    iops = None
    
    try:
        with open(filepath, 'r') as f:
            lines = f.readlines()
            
        summary_idx = -1
        for i, line in enumerate(lines):
            if "Summary of all tests:" in line:
                summary_idx = i
                break
                
        if summary_idx != -1 and len(lines) > summary_idx + 2:
            # line 0 is 'Summary of all tests:'
            # line 1 is headers (Operation Max(MiB) ...)
            # line 2 is the data row
            headers = lines[summary_idx + 1].split()
            data = lines[summary_idx + 2].split()
            
            # Map headers to indices just to be safe, though they are usually static
            try:
                mean_bw_idx = headers.index("Mean(MiB)")
                mean_iops_idx = headers.index("Mean(OPs)")
                
                # Operation is 1st column, so offset by 1 is usually not needed as headers shift
                bw = float(data[mean_bw_idx])
                iops = float(data[mean_iops_idx])
            except (ValueError, IndexError):
                pass
                
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        
    return {
        "fs_type": fs_type,
        "job_id": job_id,
        "scenario": scenario,
        "api": api,
        "run_id": run_id,
        "bw_MiB": bw,
        "iops": iops
    }

def load_all_ior_results(base_path):
    base_dir = Path(base_path)
    summary_files = list(base_dir.rglob("*_summary.txt_*"))
    
    results = []
    for f in summary_files:
        res = parse_ior_summary(f)
        if res is not None and res["bw_MiB"] is not None:
            results.append(res)
            
    df = pd.DataFrame(results)
    return df

# Load the data only for scratch
ior_results_df = load_all_ior_results(IOR_RESULTS_DIR)

if not ior_results_df.empty:
    avg_performance_df = ior_results_df.groupby(
        ["fs_type", "scenario", "api"]
    )[["bw_MiB", "iops"]].mean().reset_index()
    
    # print(f"Loaded {len(ior_results_df)} total parses.")
    # print("\n--- Averaged Performance Results ---")
    # print(avg_performance_df)
else:
    print("No valid results found. Double check the results_base_path.")

###  Compute paper table

In [14]:
def ensure_polars_df(df):
    if isinstance(df, pl.DataFrame):
        return df
    if isinstance(df, pd.DataFrame):
        # evita pl.from_pandas(...) e quindi evita la dipendenza da pyarrow
        return pl.DataFrame({col: df[col].tolist() for col in df.columns})
    raise TypeError(f"Unsupported dataframe type: {type(df)}")

avg_performance_pl = ensure_polars_df(avg_performance_df)

if "api" in avg_performance_pl.columns and "backend" not in avg_performance_pl.columns:
    avg_performance_pl = avg_performance_pl.rename({"api": "backend"})

real_perf_pl = (
    avg_performance_pl
    .group_by(["scenario", "backend"])
    .agg([
        pl.col("bw_MiB").mean().alias("real_mean_bw_MiB"),
        pl.col("iops").mean().alias("real_mean_iops"),
    ])
)

efficiency_comparison = (
    df_summary
    .join(real_perf_pl, on=["scenario", "backend"], how="inner")
    .with_columns([
        pl.col("mean_MBps_per_watt").alias("diskio_MBps_per_watt"),
        pl.col("mean_iops_per_watt").alias("diskio_iops_per_watt"),
        (pl.col("real_mean_bw_MiB") / pl.col("mean_power_W")).alias("IOR_MBps_per_watt"),
        (pl.col("real_mean_iops") / pl.col("mean_power_W")).alias("IOR_iops_per_watt"),
        (pl.col("real_mean_bw_MiB") / pl.col("mean_MBps")).alias("bw_rateo"),
        (pl.col("real_mean_iops") / pl.col("mean_iops")).alias("iops_rateo"),
    ])
    .with_columns([
        pl.when(pl.col("IOR_MBps_per_watt").is_finite()).then(pl.col("IOR_MBps_per_watt")).otherwise(0.0).alias("IOR_MBps_per_watt"),
        pl.when(pl.col("IOR_iops_per_watt").is_finite()).then(pl.col("IOR_iops_per_watt")).otherwise(0.0).alias("IOR_iops_per_watt"),
        pl.when(pl.col("bw_rateo").is_finite()).then(pl.col("bw_rateo")).otherwise(0.0).alias("bw_rateo"),
        pl.when(pl.col("iops_rateo").is_finite()).then(pl.col("iops_rateo")).otherwise(0.0).alias("iops_rateo"),
    ])
    .select([
        pl.col("backend"),
        pl.col("scenario"),
        pl.col("diskio_MBps_per_watt").round(3),
        pl.col("IOR_MBps_per_watt").round(3),
        pl.col("diskio_iops_per_watt").round(3),
        pl.col("IOR_iops_per_watt").round(3),
        pl.col("bw_rateo").round(3),
        pl.col("iops_rateo").round(3),
        pl.col("mean_power_W").round(2),
    ])
    .sort(["backend", "scenario"])
)

In [15]:
#efficiency_comparison

### Show the results

In [16]:
with pl.Config(tbl_rows=-1):
    print(avg_performance_df)
    print(efficiency_comparison)


    fs_type                  scenario    api    bw_MiB         iops
0   scratch              read_rand_4k  MPIIO     5.133     1313.712
1   scratch              read_rand_4k  POSIX     8.952     2291.677
2   scratch               read_seq_4k  MPIIO  7835.855  2005978.828
3   scratch               read_seq_4k  POSIX   344.586    88213.783
4   scratch               read_seq_4m  MPIIO  8160.559     2040.140
5   scratch               read_seq_4m  POSIX  5708.907     1427.225
6   scratch             write_rand_4k  MPIIO   361.711    92597.754
7   scratch             write_rand_4k  POSIX     4.781     1224.118
8   scratch              write_seq_4k  MPIIO  3488.393   893028.800
9   scratch              write_seq_4k  POSIX     7.936     2031.570
10  scratch              write_seq_4m  MPIIO  6231.806     1557.951
11  scratch              write_seq_4m  POSIX  3909.138      977.284
12  scratch  writeread_rand_4k_shared  MPIIO     0.113       29.014
13  scratch  writeread_rand_4k_shared  POSIX    